# Assignment 2

Deadline: 25.03.2025, 12:00 CET

<Add your name, student-id and emal address>

## 1. Linearization of Turnover

**(15 points)**

Turnover constraints are used to limit the amount of change in portfolio weights between periods, helping to manage transaction costs and maintain portfolio stability.

Your task is to implement a method `linearize_turnover_constraint` for the class `QuadraticProgram`, which modifies the quadratic programming problem to incorporate a linearized turnover constraint. This will involve updating the objective function coefficients, equality and inequality constraints, as well as the lower and upper bounds of the problem. 

Additionally, complete the example provided below to demonstrate that your method functions correctly.

In class, we discussed a solution that involved augmenting the dimensionality by a factor of three. Here, you are asked to implement an alternative method that involves a two-fold increase in dimensions. If you are unable to implement the two-fold method, you may proceed with the three-fold approach.

### Function Parameters:
- `x_init` (np.ndarray): The initial portfolio weights.
- `to_budget` (float, optional): The maximum allowable turnover. Defaults to `float('inf')`.

### Steps for Function Implementation:

As discussed in the lecture, introduce auxiliary variables and augment the matrices/vectors used for optimization.

- **Objective Function Coefficients**:  
  Pad the existing objective function coefficients (`P` and `q`) to accommodate the new variables introduced by the turnover constraint.  
  *Note*: "Padding" refers to adding extra elements (typically zeros) to an array or matrix to increase its size to a desired shape.

- **Equality Constraints**:  
  Pad the existing equality constraint matrix (`A`) to account for the new variables.

- **Inequality Constraints**:  
  Pad the existing inequality constraint matrix ('G') and vector ('h') and further add a new inequality constraint row to incorporate the turnover constraint.  

- **Lower and Upper Bounds**:  
  Pad the existing lower (`lb`) and upper (`ub`) bounds to accommodate the new variables.

- **Update Problem Data**:  
  Overwrite the original problem data in the `QuadraticProgram` class with the updated matrices and vectors to include the linearized turnover constraint.

In [21]:
# Import standard libraries
import types
import os
import sys

# Import third-party libraries
import numpy as np
import pandas as pd

# Import local modules
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))  # Change this path if needed
src_path = os.path.join(project_root, "src")
if project_root not in sys.path:
    sys.path.append(project_root)
if src_path not in sys.path:
    sys.path.append(src_path)


sys.path.append(project_root)
sys.path.append(src_path)

from estimation.covariance import Covariance
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.quadratic_program import QuadraticProgram
from helper_functions import load_data_msci

In [22]:
def linearize_turnover_constraint(self, x_init: np.ndarray, to_budget=float('inf')) -> None:
        '''
        Linearize the turnover constraint in the quadratic programming problem.

        This method modifies the quadratic programming problem to include a linearized turnover constraint.

        Parameters:
        -----------
        x_init : np.ndarray
            The initial portfolio weights.
        to_budget : float, optional
            The maximum allowable turnover. Defaults to float('inf').

        Notes:
        ------
        - The method updates the problem's objective function coefficients, inequality constraints,
        equality constraints, and bounds to account for the turnover constraint.
        - The original problem data is overridden with the updated matrices and vectors.

        Examples:
        ---------
        >>> qp = QuadraticProgram(P, q, G, h, A, b, lb, ub, solver='cvxopt')
        >>> qp.linearize_turnover_constraint(x_init=np.array([0.1, 0.2, 0.3]), to_budget=0.05)
        '''
        # Dimensions
        n = len(self.problem_data.get('q'))
        m = 0 if self.problem_data.get('G') is None else self.problem_data.get('G').shape[0]

        # Update the coefficients of the objective function
        P_old = self.problem_data.get('P')
        q_old = self.problem_data.get('q')

        P = np.block([
                [P_old, np.zeros((n, n))],
                [np.zeros((n,n)), np.zeros((n, n))]
        ])

        q = np.concatenate([q_old, np.zeros(n)])

        # Update the equality constraints
        # The equality constratints (Ax = b) represent the "fully invested" constraint. We need to pad A so it ignores the y variables
        A_old = self.problem_data.get('A')
        A = np.hstack([A_old, np.zeros((A_old.shape[0], n))])

        # Update the inequality constraints -> stack original constraints on top of our new linearization constraints
        G_old = self.problem_data.get('G')
        h_old = self.problem_data.get('h')

        if G_old is None:
            G_old = np.empty((0, n))
            h_old = np.empty(0)

        if A_old is None:
              A_old = np.empty((0, n))
        
        x_val = x_init.values if hasattr(x_init, 'values') else x_init.flatten()

        I = np.eye(n)

        # Combine: 
        # [G_old, 0]  <-- Original constraints
        # [ I,  -I]  <-- x - y <= x_init
        # [-I,  -I]  <-- -x - y <= -x_init
        # [ 0,   1]  <-- sum(y) <= budget
        G = np.block([
            [G_old,           np.zeros((m, n))],
            [I,               -I],
            [-I,              -I],
            [np.zeros((1, n)), np.ones((1, n))]
        ])

        h = np.concatenate([
            h_old.flatten(), 
            x_init.values.flatten(), 
            -x_init.values.flatten(), 
            np.array([to_budget])
        ])

        # Update lower and upper bounds -> x keep the same bounds, but are non-negative
        lb_old = self.problem_data.get('lb')
        ub_old = self.problem_data.get('ub')

        # y_i must be >= 0. Upper bound for y can be 1 (or inf)
        lb = np.concatenate([lb_old, np.zeros(n)])
        ub = np.concatenate([ub_old, np.ones(n)])

        # Override the original matrices (notice: b does not change)
        self.update_problem_data({
            'P': P,
            'q': q,
            'G': G,
            'h': h,
            'A': A,
            'lb': lb,
            'ub': ub,
        })

        return None

## Demo

#### Create P and q

In [23]:
# Load the msci country index data
N = 10
data = load_data_msci(path = '../data/', n=N)
X = data['return_series']

# Compute the vector of expected returns (mean returns)
q = ExpectedReturn(method='geometric').estimate(X=X, inplace=False)

# Compute the covariance matrix
P = Covariance(method='pearson').estimate(X=X, inplace=False)

q, P

(AT    0.000130
 AU    0.000288
 BE    0.000047
 CA    0.000269
 CH    0.000149
 DE    0.000151
 DK    0.000429
 ES    0.000128
 FI    0.000145
 FR    0.000199
 dtype: float64,
           AT        AU        BE        CA        CH        DE        DK  \
 AT  0.000239  0.000054  0.000125  0.000075  0.000097  0.000138  0.000097   
 AU  0.000054  0.000104  0.000039  0.000030  0.000035  0.000041  0.000041   
 BE  0.000125  0.000039  0.000175  0.000064  0.000104  0.000137  0.000093   
 CA  0.000075  0.000030  0.000064  0.000130  0.000058  0.000087  0.000053   
 CH  0.000097  0.000035  0.000104  0.000058  0.000120  0.000121  0.000086   
 DE  0.000138  0.000041  0.000137  0.000087  0.000121  0.000202  0.000105   
 DK  0.000097  0.000041  0.000093  0.000053  0.000086  0.000105  0.000151   
 ES  0.000150  0.000044  0.000138  0.000081  0.000116  0.000164  0.000100   
 FI  0.000140  0.000050  0.000136  0.000091  0.000126  0.000180  0.000119   
 FR  0.000143  0.000045  0.000142  0.000084  0.000122

### Create some constraints, instantiate an object of class QuadraticProgram, and add the method linearize_turnover_constraint to the instance.

In [24]:
# Instantiate the constraints with only the budget and long-only constraints
constraints = Constraints(ids = X.columns.tolist())
constraints.add_budget(rhs=1, sense='=')
constraints.add_box(lower=0.0, upper=1.0)
GhAb = constraints.to_GhAb()

# Create a quadratic program and linearize the turnover constraint
qp = QuadraticProgram(
    P = P.to_numpy(),
    q = q.to_numpy() * 0,
    G = GhAb['G'],
    h = GhAb['h'],
    A = GhAb['A'],
    b = GhAb['b'],
    lb = constraints.box['lower'].to_numpy(),
    ub = constraints.box['upper'].to_numpy(),
    solver = 'cvxopt',
)

# Add the linearized turnover constraint method to the instance of class QuadraticProgram
qp.linearize_turnover_constraint = types.MethodType(linearize_turnover_constraint, qp)


### Add a turnover limit of 50%. Solve the problem and check whether the turnover constraint is respected.

In [25]:
# Prepare initial weights
x_init = pd.Series([1/X.shape[1]]*X.shape[1], index=X.columns)

# Add the linearized turnover constraint
qp.linearize_turnover_constraint(x_init=x_init, to_budget=0.5)

# Check the updated problem data dimensions (should be (2N, 2N))
qp.problem_data['P'].shape

# Solve the problem
qp.solve()

# Check the turnover
solution = qp.results.get('solution')
ids = constraints.ids
weights = pd.Series(solution.x[:len(ids)], index=ids)

print("Turnover:")
print(np.abs(weights - x_init).sum())


Turnover:
0.49980829943877114


### Notes for me

1. The "Alphabet Soup" (The Variables)

In a Quadratic Program (QP), we are just giving the computer a list of rules to follow while it searches for the "best" portfolio.

- x (The Weights): These are your decisions. "How much Apple stock should I own? How much Tesla?"

- P (Risk): This matrix represents Risk. It tells the computer which stocks "swing" together. A big P value means "this is a bumpy ride."

- q (Return): This represents Expected Profit. It tells the computer which stocks are likely to make money.

- A and b (The "Must-Haves"): These are Equality Constraints. For example: "The total of all my stocks must equal 100% of my cash."

- G and h (The "Limits"): These are Inequality Constraints. For example: "I cannot have more than 20% in any one stock."

2. The Problem: What is "Turnover"?

Turnover is just a fancy word for "How much did I trade?"

If you owned 10% Apple yesterday and now you want 15%, your turnover for that stock is 5%. If you owned 10% and now want 5%, your turnover is also 5%.

The computer struggles because turnover uses Absolute Values (∣x−x init∣).

- In math, absolute values create a "V" shape.

- Standard optimization solvers are like cars that can only drive on straight roads or smooth curves. They "crash" when they hit the sharp point at the bottom of a "V."

3. The "Two-Fold" Trick (Linearization)

Since the computer can't handle the "V" shape, we "trick" it by adding a Helper Variable (y).

Think of y as a Shadow Variable. Instead of asking the computer to calculate the absolute value (the "V"), we give it two simple, straight-line rules for every stock:

- Rule A: "Your change (y) must be at least as big as (NewWeight−OldWeight)."

- Rule B: "Your change (y) must be at least as big as (OldWeight−NewWeight)."

By forcing y to be bigger than both the positive and negative difference, y is effectively forced to be the absolute value.

Why call it "Two-Fold"? Because if you have 50 stocks, you now have 50 "Decision" variables (x) and 50 "Shadow" variables (y). Your problem just got twice as big (two-fold).

4. Why we "Pad" the Matrices

When we add these shadow variables (y), the computer's "map" gets bigger.

- The Original Map (x): Just the stocks.

- The New Map (x and y): Stocks + Shadows.

When we "pad" P or q with zeros, we are telling the computer: "Calculate the risk and return based on the stocks (x), but don't worry about the risk of the shadows (y)." We only use the shadows to check the turnover budget.